In [10]:
import re
from collections import defaultdict

def analyze_failed_logins(log_content):
    """
    Parses a log file content to count failed login attempts per IP address.

    Args:
        log_content (str): A multi-line string representing the log file.

    Returns:
        dict: A dictionary where keys are IP addresses and values are the count of failed attempts.
    """
    print("Starting login log analysis...")
    failed_attempts = defaultdict(int)

    # Regex to find an IPv4 address. Adjust if your logs contain IPv6.
    ip_pattern = re.compile(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b')

    # Keywords indicating a failed login attempt. This might need customization
    # based on the specific log source (e.g., SSH, FTP, web application).
    failed_keywords = ['Failed password', 'Authentication failure', 'failed login']

    log_lines = log_content.strip().split('\n')

    for i, line in enumerate(log_lines):
        # Check if the line contains any of the failed login keywords
        if any(keyword in line for keyword in failed_keywords):
            # Extract all IP addresses from the line
            found_ips = ip_pattern.findall(line)
            for ip in found_ips:
                # Exclude internal/loopback IPs if necessary, or refine based on context
                if not ip.startswith('127.') and not ip.startswith('192.168.') and not ip.startswith('10.'):
                    failed_attempts[ip] += 1
                    print(f"  - Failed attempt from {ip} detected in Line {i+1}: '{line}'")

    if not failed_attempts:
        print("No failed login attempts found in the log.")
    else:
        print("\nAnalysis complete. Failed login attempts per IP:")
        for ip, count in failed_attempts.items():
            print(f"  IP: {ip:<15} Failed Attempts: {count}")

    print("\n")
    return dict(failed_attempts)

# --- Demonstration ---
print("--- Failed Login Attempt Analysis Demonstration ---\n")

# Sample login log content
sample_login_log = """
Jan 1 00:00:01 host sshd[123]: Accepted password for user from 192.168.1.100 port 12345 ssh2
Jan 1 00:00:02 host sshd[124]: Failed password for invalid user test from 203.0.113.1 port 54321 ssh2
Jan 1 00:00:03 host sshd[125]: Failed password for user admin from 203.0.113.1 port 54322 ssh2
Jan 1 00:00:04 host sshd[126]: Accepted password for user root from 192.168.1.101 port 12346 ssh2
Jan 1 00:00:05 host sshd[127]: Failed password for user guest from 203.0.113.2 port 54323 ssh2
Jan 1 00:00:06 host sshd[128]: Authentication failure for user baduser from 203.0.113.3 port 54324 ssh2
Jan 1 00:00:07 host sshd[129]: Failed password for invalid user admin from 203.0.113.1 port 54325 ssh2
Jan 1 00:00:08 host kernel: other log message
Jan 1 00:00:09 host sshd[130]: Failed password for user root from 203.0.113.2 port 54326 ssh2
Jan 1 00:00:10 host sshd[131]: Failed password for invalid user john from 203.0.113.4 port 54327 ssh2
"""

# Analyze the sample log
failed_login_counts = analyze_failed_logins(sample_login_log)

--- Failed Login Attempt Analysis Demonstration ---

Starting login log analysis...
  - Failed attempt from 203.0.113.1 detected in Line 2: 'Jan 1 00:00:02 host sshd[124]: Failed password for invalid user test from 203.0.113.1 port 54321 ssh2'
  - Failed attempt from 203.0.113.1 detected in Line 3: 'Jan 1 00:00:03 host sshd[125]: Failed password for user admin from 203.0.113.1 port 54322 ssh2'
  - Failed attempt from 203.0.113.2 detected in Line 5: 'Jan 1 00:00:05 host sshd[127]: Failed password for user guest from 203.0.113.2 port 54323 ssh2'
  - Failed attempt from 203.0.113.3 detected in Line 6: 'Jan 1 00:00:06 host sshd[128]: Authentication failure for user baduser from 203.0.113.3 port 54324 ssh2'
  - Failed attempt from 203.0.113.1 detected in Line 7: 'Jan 1 00:00:07 host sshd[129]: Failed password for invalid user admin from 203.0.113.1 port 54325 ssh2'
  - Failed attempt from 203.0.113.2 detected in Line 9: 'Jan 1 00:00:09 host sshd[130]: Failed password for user root from 203.